In [1]:
!pip install -q torch torchvision \transformers \tqdm \pillow \ boto3

In [2]:
import time
import logging
import os
import torch
import boto3
from torch.utils.data import DataLoader
from transformers import ViltProcessor

In [3]:
from models.vilt_adapter import ViLTAdapter
from data.clevr_baseline_data import load_answer_vocab, CLEVRBaselineDatasetS3, vilt_collate_fn
from services.checkpoint_service import CheckpointManager

In [4]:
logger = logging.getLogger(__name__)

In [12]:
CONFIG = {
    
"seed":42,
    
# S3
"s3_bucket":           "clevr-curriculum",
"s3_images_prefix":    "dataset/images",
"s3_questions_prefix": "dataset/questions",

# Answer vocab
"answer_vocab_path": "data/answer_vocab.json",

# Dataset
"max_question_length": 32,
"max_train_samples":   320000,

# Model
"model_name":          "dandelin/vilt-b32-finetuned-vqa",
"learning_rate":       5e-5,
"freeze_backbone":     False,
"device":              "cuda",

# Training
"batch_size":          32,
"max_steps":           100,   
"log_every":           50,
"save_every":          50,

# Checkpoints
"run_name":            "baseline_training_run_5",
"checkpoint_prefix":   "checkpoints/baseline",
"resume_training":     True
}

In [8]:
def set_seed(seed=42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
def _setup_logging():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(name)-28s  %(levelname)-7s  %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

In [9]:
def load_latest_checkpoint_from_s3(config, model):
    """
    Resume training from the most recent checkpoint stored in S3.

    Returns: step to resume from
    """
    s3 = boto3.client("s3")
    bucket = config["s3_bucket"]
    prefix = config["checkpoint_prefix"]
    run_name = config.get("run_name", "baseline_run")

    # Scope the search to the correct run
    full_prefix = f"{prefix}/{run_name}/"

    response = s3.list_objects_v2(
        Bucket=bucket,
        Prefix=full_prefix
    )
    if "Contents" not in response:
        logger.info(f"No checkpoint found at {full_prefix}. Starting fresh.")
        return 0

    files = [obj["Key"] for obj in response["Contents"] if "latest" in obj["Key"]]
    if len(files) == 0:
        logger.info(f"No latest checkpoint found under {full_prefix}. Starting fresh.")
        return 0

    latest_key = files[0]
    local_path = "latest_checkpoint.pt"
    logger.info(f"Downloading checkpoint {latest_key}")
    s3.download_file(bucket, latest_key, local_path)

    checkpoint = torch.load(local_path, map_location=model.device)
    model.model.load_state_dict(checkpoint["model_state_dict"])
    model.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    step = checkpoint.get("step", 0)
    logger.info(f"Resumed training from step {step}")
    return step

In [10]:
def main():
    """
    End-to-end training entry point for the ViLT baseline model.

    Execution order:
      1. Config setup
      2. Processor + answer vocab loading
      3. Dataset & DataLoader construction (train and  val, streamed from S3)
      4. ViLTAdapter (model + optimizer) initialisation
      5. Optional resume from the latest S3 checkpoint
      6. Step-based training loop 
      7. Full validation pass over the val split
      8. Final checkpoint upload to S3
    """
    _setup_logging()
    
    config = CONFIG
    processor = ViltProcessor.from_pretrained(config["model_name"])
    answer2id = load_answer_vocab(config)
    num_classes = len(answer2id)

    # 1) Datasets
    logger.info("Initializing baseline train dataset...")
    train_dataset = CLEVRBaselineDatasetS3(
        bucket=config["s3_bucket"],
        images_prefix=config["s3_images_prefix"],
        questions_prefix=config["s3_questions_prefix"],
        processor=processor,
        filename="CLEVR_train_baseline_questions.json",  # Training split question file
        answer2id=answer2id,
        max_length=config["max_question_length"],
        max_samples=config["max_train_samples"], 
    )

    logger.info("Initializing baseline validation dataset...")
    val_dataset = CLEVRBaselineDatasetS3(
        bucket=config["s3_bucket"],
        images_prefix=config["s3_images_prefix"],
        questions_prefix=config["s3_questions_prefix"],
        processor=processor,
        filename="CLEVR_val_baseline_questions.json",  # Validation split question file
        answer2id=answer2id,
        max_length=config["max_question_length"],
    )

    train_loader = DataLoader(
        train_dataset, 
        batch_size=config["batch_size"], 
        shuffle=True,  # shuffle=True for train 
        collate_fn=vilt_collate_fn
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=config["batch_size"], 
        shuffle=False, 
        collate_fn=vilt_collate_fn
    )

    # 2) Model
    logger.info(f"Loaded Model: {config['model_name']} (classes={num_classes})")
    model = ViLTAdapter(
        model_name=config["model_name"],
        num_labels=num_classes,
        learning_rate=config["learning_rate"],
        device=config["device"],
        freeze_backbone=config["freeze_backbone"],
    )
    # Checkpoint Manager
    checkpoint_manager = CheckpointManager(
        bucket=config["s3_bucket"],
        run_name=config.get("run_name", "baseline_run"),
        prefix=config.get("checkpoint_prefix", "checkpoints/baseline")
    )

    # Resume Training
    step = load_latest_checkpoint_from_s3(config, model)
    # 3) Training Loop
    logger.info("=" * 60)
    logger.info(f"Starting Baseline Training for {len(train_dataset)} samples")
    logger.info("=" * 60)

    train_start = time.time()

    train_start = time.time()

    train_iter = iter(train_loader)

    while step < config["max_steps"]:
        # Fetch the next batch
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)
    
        step_start = time.time()
    
        out = model.train_step(batch) # Forward pass + backward pass + optimizer step
        loss = out["loss"]
    
        step_time = time.time() - step_start

        # Periodic training log
        if step % config["log_every"] == 0:
            logger.info(f"[ Step {step}] loss={loss:.4f} | {step_time:.2f}s")

        # Periodic intermediate checkpoint
        if step > 0 and step % config["save_every"] == 0:
            logger.info(f"Saving intermediate baseline checkpoint at step {step}...")
            checkpoint_manager.save_baseline(
                step=step,
                model=model,
                save_numbered=True
            )
    
        step += 1
    
    total_time = time.time() - train_start
    logger.info(f"Training complete. Time taken: {total_time:.2f}s")

    # 4) Validation Pass
    logger.info("-" * 45)
    logger.info("[Validation END] Running...")
    val_start = time.time()
    # Accumulators for computing epoch-level metrics
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    num_batches = 0

    for batch in val_loader:
        out = model.validation_step(batch)  # Forward pass only
        logits = out["logits"]
        labels = out["labels"]

        # Weight loss by batch size so the final average is sample-weighted
        total_loss += out["loss"] * logits.size(0)
        preds = logits.argmax(dim=-1)
        # Labels may be soft or hard - normalise to hard
        if labels.dim() == 2:
            labels = labels.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += logits.size(0)
        num_batches += 1

    # Guard against empty val set with max(1) to prevent division by zero
    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)
    val_time = time.time() - val_start

    logger.info(
        f"[Validation END] loss={avg_loss:.4f} | accuracy={accuracy:.4f} | "
        f"{total_samples} samples, {num_batches} batches | {val_time:.1f}s"
    )
    logger.info("-" * 45)
    # 5) Final Checkpoint
    logger.info("Saving final baseline checkpoint...")
    checkpoint_manager.save_baseline(
        step=step,
        model=model,
        accuracy=accuracy,
    )


In [ ]:
if __name__ == "__main__":
    main()

2026-04-03 23:40:42  httpx                         INFO     HTTP Request: GET https://huggingface.co/api/models/dandelin/vilt-b32-finetuned-vqa/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-04-03 23:40:42  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-04-03 23:40:42  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-04-03 23:40:42  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-04-03 23:40:42  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 

Loading dataset/questions/CLEVR_train_baseline_questions.json from S3...
Loaded 281437 questions from dataset/questions/CLEVR_train_baseline_questions.json


2026-04-03 23:41:05  __main__                      INFO     Initializing baseline validation dataset...


Loading dataset/questions/CLEVR_val_baseline_questions.json from S3...
Loaded 40780 questions from dataset/questions/CLEVR_val_baseline_questions.json


2026-04-03 23:41:10  __main__                      INFO     Loaded Model: dandelin/vilt-b32-finetuned-vqa (classes=28)
2026-04-03 23:41:10  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-03 23:41:10  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dandelin/vilt-b32-finetuned-vqa/d0a1f6ab88522427a7ae76ceb6e1e1e7b68a1d08/config.json "HTTP/1.1 200 OK"
You passed `num_labels=28` which is incompatible to the `id2label` map of length `3129`.
2026-04-03 23:41:10  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-04-03 23:41:10  httpx                         INFO     HTTP Request: HEAD https://huggingface.co/dandelin/vilt-b32-finetuned-vqa/resolve/main/model.safetensors.index.json "HT

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

ViltForQuestionAnswering LOAD REPORT from: dandelin/vilt-b32-finetuned-vqa
Key                                          | Status     |                                                                                             
---------------------------------------------+------------+---------------------------------------------------------------------------------------------
vilt.embeddings.text_embeddings.position_ids | UNEXPECTED |                                                                                             
classifier.3.weight                          | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3129, 1536]) vs model:torch.Size([28, 1536])
classifier.3.bias                            | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3129]) vs model:torch.Size([28])            

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISMATCH:	ckpt weights were loaded, b